In [1]:
import os

In [2]:
%pwd

'd:\\ML-Projects\\End-to-end-Machine-Learning-World-Development-Measurement-Clustering-Analysis-with-MLFlow\\notebooks'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\ML-Projects\\End-to-end-Machine-Learning-World-Development-Measurement-Clustering-Analysis-with-MLFlow'

In [25]:
## Preparing entity

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    transformed_data_path: Path 
    model_params: dict
    TRAINING_REPORT: Path

In [26]:
import importlib
import wdmproject.utils.common as common
importlib.reload(common)

<module 'wdmproject.utils.common' from 'D:\\ML-Projects\\End-to-end-Machine-Learning-World-Development-Measurement-Clustering-Analysis-with-MLFlow\\src\\wdmproject\\utils\\common.py'>

In [27]:
import wdmproject.utils.common as common

print(dir(common))

['Any', 'BoxValueError', 'ConfigBox', 'Path', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'calinski_harabasz_score', 'create_directories', 'davies_bouldin_score', 'get_size', 'is_standard_column', 'joblib', 'json', 'load_bin', 'load_json', 'logger', 'os', 're', 'read_yaml', 'save_bin', 'save_json', 'silhouette_score', 'standardize_column_name', 'yaml']


In [28]:
## Configuration

from wdmproject.constants import *
from wdmproject.utils.common import read_yaml, create_directories

In [29]:
## Configuration manager class
class ConfigurationManager:
    def __init__(
        self,
        config_filepath: Path = CONFIG_FILE_PATH,
        params_filepath: Path = PARAMS_FILE_PATH,
        schema_filepath: Path = SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        
        create_directories([self.config.artifacts_root])

    
    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.model_training

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=Path(config.root_dir),
            transformed_data_path=Path(config.transformed_data_path),
            model_params=params,
            TRAINING_REPORT=Path(config.TRAINING_REPORT)
        )

        return model_trainer_config

In [30]:
# Components


import os
import json
from wdmproject import logger
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, MeanShift
from sklearn.mixture import GaussianMixture
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import pickle, joblib
#from wdmproject.utils.common import evaluate_models

In [31]:
# Component

from scipy import cluster
from sklearn.cluster import dbscan
#from wdmproject.utils.common import evaluate_models

class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config
        self.data = None
        self.training_report = {}
        

    ## Load transformed data for training
    def load_transformed_data(self):
        try:
            logger.info("Loading transformed PCA data")

            X = joblib.load(self.config.transformed_data_path)           
            
            logger.info(f"Transformed data loaded: {X.shape}")
            

            nan_count = np.isnan(X).sum()
            ## NaN Values
            if nan_count > 0:
                logger.error(f"NaN Values found in transformed data: {nan_count}")
                raise ValueError(f"NaN values detected: {nan_count}")
            
            ## Infinite Values
            inf_count = np.isinf(X).sum()
            if inf_count > 0:
                logger.error(f"Infinite Values found in transformed data: {inf_count}")
                raise ValueError(f"Infinite values detected: {inf_count}")


            self.training_report['dataset_load'] = {
                 "status" : "Success",
                 "rows" : X.shape[0],
                 "columns" : X.shape[1],
                 "nan_values" : int(nan_count),
                 "inf_values" : int(inf_count)

            }
            
            return X

        except Exception as e:
            logger.error(f"Failed loading transformed data: {e}")
            self.training_report['dataset_load'] = {
                 "status" : "Failed",
                 "error_msg" : "Error Loading Dataset.",
                 "error" : str(e),
            }
            raise e
        

    def evaluate_models(self,model, X, model_name):
        try:
            labels = model.fit_predict(X)

            if(len(set(labels))) > 1:
                sil_score = silhouette_score(X, labels) 
                #dv_score = davies_bouldin_score(X, labels)
                #ch_score = calinski_harabasz_score(X, labels)

            else:
                sil_score = -1

            self.training_report[model_name] = {
                "silhouete_score": float(sil_score),        
                "n_clusters": int(len(set(labels)))    
            }

            logger.info(f"Silhouete_score {sil_score}")

            return sil_score, labels

        except Exception as e:
        
            logger.error(f"training failed: {e}")

            self.training_report["model_name"] = {
                "status": "Failed",
                "error": str(e)
            }

            return -1, None

    ## Training all clustering models
    def train_models(self, X):
        try:
            params = self.config.model_params

            models = {

                "k_means": KMeans(
                    n_clusters=params.kmeans.n_clusters,
                    random_state=params.kmeans.random_state,
                    n_init=params.kmeans.n_init
                ),

                "hierarchical": AgglomerativeClustering(
                    n_clusters=params.hierarchical.n_clusters,
                    linkage=params.hierarchical.linkage
                ),

                "dbscan": DBSCAN(
                    eps=params.dbscan.eps,
                    min_samples=params.dbscan.min_samples
                ),

                "meanshift": MeanShift(
                    cluster_all=params.meanshift.cluster_all                
                ),

                "gmm": GaussianMixture(
                    n_components=params.gmm.n_components,
                    random_state=params.gmm.random_state 
                )
            }

            best_score = -1
            best_model = None
            best_model_name = None
            best_labels= None

            for name, model in models.items():

                logger.info(f"Training {name} model")

                score, labels = self.evaluate_models(model, X, name)

                if score > best_score:

                    best_score = score
                    best_model = model
                    best_model_name = name
                    best_labels=labels
            
            self.training_report["best_model"] = best_model_name
            self.training_report["best_score"] = float(best_score)
            self.training_report["clusters"] = len(set(best_labels))

            return best_model
        
        except Exception as e:
            logger.error(f"Training Models Failed : {str(e)}")
            raise e
            


    ## Saving Training Report 
    def save_training_report(self):

        report_path = self.config.TRAINING_REPORT

        with open(report_path, "w") as f:

            json.dump(self.training_report, f, indent=4)

        logger.info(f"Model Training report saved at {report_path}")



    ## Initialising Model Trainer Pipeline 

    def initiate_model_trainer(self):
        try: 
            
            ## Intiatining Model Trainer stage

            self.training_report["stage_metadata"] = {
                "stage_name" : "Model Trainer",
                "stage_status" : "Running",
                "start_time" : str(datetime.now())
            }

            logger.info("Initiating Model Trainer Stage.")

            ## Saving start_time
            start_time = datetime.now()


            ## Loading Dataset
            X = self.load_transformed_data()
            print(X)
            ## model_trainer
            self.train_models(X)

            ## Stage Success
            self.training_report["stage_metadata"]["stage_status"] = "Success"
            self.training_report["stage_metadata"]["end_time"] = str(datetime.now())

            ## Calculating Stage Duration
            stage_duration = (datetime.now() - start_time).total_seconds()
            self.training_report["stage_metadata"]["stage_duration"] = stage_duration
            
            # Saving Transformations Report JSON
            self.save_training_report()

            logger.info("Model Training Completed Successfully.")

        except Exception as e:
            
            logger.error(f"Model Training Error : {e}")

            end_time = datetime.now()

            self.training_report["stage_metadata"]["stage_status"] = "Failed"
            self.training_report["stage_metadata"]["end_time"] = str(end_time)
            self.training_report["stage_metadata"]["error"] = str(e)

            self.save_training_report()

            raise e


In [32]:
# Pipeline
## Model Trainer Pipeline
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.initiate_model_trainer()
    logger.info(f"Moder Trainer Pipeline Completed Successfully.")
except Exception as e:
    logger.error(f"Model Trainer Pipeline Failed: {e}")
    raise e

[2026-04-01 16:34:27,407: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-01 16:34:27,416: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-01 16:34:27,429: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-04-01 16:34:27,433: INFO: common: created directory at: artifacts]
[2026-04-01 16:34:27,436: INFO: common: created directory at: artifacts/model_trainer/models]
[2026-04-01 16:34:27,438: INFO: 2484883155: Initiating Model Trainer Stage.]
[2026-04-01 16:34:27,440: INFO: 2484883155: Loading transformed PCA data]
[2026-04-01 16:34:27,474: INFO: 2484883155: Transformed data loaded: (2704, 13)]
[[ 0.92335756  1.13049225  0.48510186 ...  0.01963775  0.37764844
  -0.41530211]
 [ 6.71429392  2.39528818 -0.77279751 ... -0.15318424 -0.10328876
   1.05268994]
 [ 5.00690759  0.23516017 -0.92358794 ... -0.82589754  0.51647231
   0.3103813 ]
 ...
 [-3.49150955 -0.72064031 -0.56940546 ... -0.22278028  0.57341638
  -0.3245304 ]
 [-1.1365866 